# Step 1: Data Exploration & Cleaning

This notebook is the **first stage** of the book recommender pipeline.

**Goal:** Turn raw book metadata into a clean dataset that downstream notebooks can use.

**Pipeline position:**
```
Raw books.csv  →  [this notebook]  →  books_cleaned.csv  →  classification / vector search
```

We keep only books with useful descriptions (25+ words) because short descriptions are too vague for NLP tasks like classification, emotion analysis, and semantic search.

## Load the raw dataset

The source data contains title, authors, description, category, ratings, and publication year. Many rows have missing values — we filter those out later.

In [1]:
12

12

In [2]:
# Load raw book metadata from CSV.
import os
import numpy as np
import pandas as pd

books = pd.read_csv(os.path.join(os.getcwd(), "datasets/books.csv"))
books.head(3)

,isbn13,isbn10,title,subtitle,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count
0,9780002005883,0002005883,Gilead,NaN,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0
1,9780002261982,0002261987,Spider's Web,A Novel,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0
2,9780006163831,0006163831,The One Tree,NaN,Stephen R. Donaldson,American fiction,http://books.google.com/books/content?id=OmQaw...,Volume Two of Stephen Donaldson's acclaimed se...,1982.0,3.97,479.0,172.0


## Filter to complete, usable records

We require non-null **description**, **page count**, **rating**, and **publication year**. Books without a description cannot be searched or classified by text.

In [3]:
# Keep only rows with complete key fields.
book_missing = books[
    books["description"].notna()
    & books["num_pages"].notna()
    & books["average_rating"].notna()
    & books["published_year"].notna()
]
print(f"Filtered from {len(books):,} to {len(book_missing):,} books")

Filtered from 6,810 to 6,507 books


## Keep descriptions rich enough for NLP

Descriptions with fewer than 25 words rarely carry enough meaning for zero-shot classification or embedding-based search. We count words and keep the longer ones.

In [4]:
# Filter to books with at least 25 words in the description.
book_missing["words_in_description"] = book_missing["description"].str.split().str.len()
book_missing_25_words = book_missing[book_missing["words_in_description"] >= 25]
print(f"Books with 25+ word descriptions: {len(book_missing_25_words):,}")

Books with 25+ word descriptions: 5,197


C:\Users\Asus\AppData\Local\Temp\ipykernel_8212\4173371872.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  book_missing["words_in_description"] = book_missing["description"].str.split().str.len()


## Prepare fields for later stages

- **`title_and_subtitle`**: A single display title for the UI.
- **`tagged_description`**: Prepends the ISBN to each description so vector search can map results back to the correct book.

In [5]:
# Build display title and tagged description for vector search.
book_missing_25_words["title_and_subtitle"] = np.where(
    book_missing_25_words["subtitle"].isna(),
    book_missing_25_words["title"],
    book_missing_25_words[["title", "subtitle"]].astype(str).agg(": ".join, axis=1),
)

book_missing_25_words["tagged_description"] = book_missing_25_words[
    ["isbn13", "description"]
].astype(str).agg(" ".join, axis=1)

book_missing_25_words[["isbn13", "title_and_subtitle", "tagged_description"]].head(2)

C:\Users\Asus\AppData\Local\Temp\ipykernel_8212\3105766352.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  book_missing_25_words["title_and_subtitle"] = np.where(
C:\Users\Asus\AppData\Local\Temp\ipykernel_8212\3105766352.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  book_missing_25_words["tagged_description"] = book_missing_25_words[


,isbn13,title_and_subtitle,tagged_description
0,9780002005883,Gilead,9780002005883 A NOVEL THAT READERS and critics...
1,9780002261982,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...


## Export cleaned dataset

This file is the shared input for **text classification**, **vector search**, and (indirectly) the **Gradio dashboard**.

In [6]:
# Save cleaned dataset for downstream notebooks.
(
    book_missing_25_words
    .drop(["subtitle", "missing_description", "age_of_book", "words_in_description"], axis=1, errors="ignore")
    .to_csv("datasets/books_cleaned.csv", index=False)
)
print("Saved datasets/books_cleaned.csv")

Saved datasets/books_cleaned.csv
